# Benchmark: LitData streaming over S3


In [ ]:
import io
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from torchvision import transforms

try:
    from litdata import StreamingDataset
    from litdata.streaming import StreamingDataLoader
except ImportError as e:
    raise ImportError('Missing dependency: litdata (pip install litdata)') from e

try:
    from PIL import Image
except ImportError as e:
    raise ImportError('Missing dependency: Pillow (pip install pillow)') from e

print('torch:', torch.__version__)


## Configuration

In [ ]:
S3_BUCKET = os.environ.get('S3_BUCKET', '')
S3_PREFIX = os.environ.get('S3_PREFIX', 'Food-11-litdata')
SPLIT = os.environ.get('FOOD11_SPLIT', 'evaluation')
S3_ENDPOINT_URL = os.environ.get('S3_ENDPOINT_URL', '')

AWS_ACCESS_KEY_ID = os.environ.get('AWS_ACCESS_KEY_ID', '')
AWS_SECRET_ACCESS_KEY = os.environ.get('AWS_SECRET_ACCESS_KEY', '')

BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '64'))
NUM_WORKERS = 4

WARMUP_BATCHES = 10
MEASURE_BATCHES = 50

LITDATA_CACHE_DIR = os.environ.get('LITDATA_CACHE_DIR', './litdata_cache')
MAX_PRE_DOWNLOAD = int(os.environ.get('LITDATA_MAX_PRE_DOWNLOAD', '32'))

if not S3_BUCKET:
    raise ValueError('S3_BUCKET env var is required')
if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise ValueError('AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY env vars are required')

print('S3_BUCKET:', S3_BUCKET)
print('S3_PREFIX:', S3_PREFIX)
print('SPLIT:', SPLIT)
print('S3_ENDPOINT_URL:', S3_ENDPOINT_URL if S3_ENDPOINT_URL else '(default)')
print('BATCH_SIZE:', BATCH_SIZE)
print('NUM_WORKERS:', NUM_WORKERS)
print('WARMUP_BATCHES:', WARMUP_BATCHES)
print('MEASURE_BATCHES:', MEASURE_BATCHES)
print('LITDATA_CACHE_DIR:', LITDATA_CACHE_DIR)
print('MAX_PRE_DOWNLOAD:', MAX_PRE_DOWNLOAD)


## Dataset (LitData StreamingDataset)

In [ ]:
Path(LITDATA_CACHE_DIR).mkdir(parents=True, exist_ok=True)

input_dir = f's3://{S3_BUCKET}/{S3_PREFIX}/{SPLIT}'

storage_options = {}
if S3_ENDPOINT_URL:
    storage_options['endpoint_url'] = S3_ENDPOINT_URL

session_options = {
    'aws_access_key_id': AWS_ACCESS_KEY_ID,
    'aws_secret_access_key': AWS_SECRET_ACCESS_KEY,
}

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def decode_sample(sample):
    img = Image.open(io.BytesIO(sample['image'])).convert('RGB')
    img = transform(img)
    return img, int(sample['label'])

def collate_fn(batch):
    xs, ys = zip(*(decode_sample(s) for s in batch))
    return torch.stack(xs), torch.tensor(ys)

dataset = StreamingDataset(
    input_dir=input_dir,
    cache_dir=LITDATA_CACHE_DIR,
    shuffle=False,
    max_pre_download=MAX_PRE_DOWNLOAD,
    storage_options=storage_options,
    session_options=session_options,
)

print('input_dir:', input_dir)


## DataLoader

In [ ]:
num_workers = NUM_WORKERS

loader = StreamingDataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=num_workers,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False,
    prefetch_factor=2,
)



## Run benchmark

In [ ]:
it = iter(loader)
t0 = time.perf_counter()
x0, y0 = next(it)
t1 = time.perf_counter()
time_to_first_batch_s = t1 - t0

for _ in range(WARMUP_BATCHES):
    try:
        _ = next(it)
    except StopIteration:
        break

num_batches = 0
num_items = 0
t_start = time.perf_counter()
for _ in range(MEASURE_BATCHES):
    try:
        x, y = next(it)
    except StopIteration:
        break
    num_batches += 1
    num_items += int(y.shape[0])
t_end = time.perf_counter()

wall_s = t_end - t_start
imgs_per_s = (num_items / wall_s) if wall_s > 0 else float('nan')
batches_per_s = (num_batches / wall_s) if wall_s > 0 else float('nan')
avg_batch_s = (wall_s / num_batches) if num_batches > 0 else None

result = {
    'num_workers': num_workers,
    'batch_size': BATCH_SIZE,
    'time_to_first_batch_s': time_to_first_batch_s,
    'measured_batches': num_batches,
    'measured_items': num_items,
    'wall_s': wall_s,
    'imgs_per_s': imgs_per_s,
    'batches_per_s': batches_per_s,
    'avg_batch_s': avg_batch_s,
}

result


## Print results

In [ ]:
print('split:', SPLIT)
print('batch_size:', BATCH_SIZE)
print('warmup_batches:', WARMUP_BATCHES, 'measure_batches:', MEASURE_BATCHES)
print()

avg_batch_s = result['avg_batch_s']
avg_batch_s_str = 'nan' if avg_batch_s is None else f"{avg_batch_s:.4f}"
print(
    'workers=', result['num_workers'],
    'imgs/s=', f"{result['imgs_per_s']:.2f}",
    'batches/s=', f"{result['batches_per_s']:.2f}",
    'first_batch_s=', f"{result['time_to_first_batch_s']:.3f}",
    'avg_batch_s=', avg_batch_s_str,
)


## Save results

In [ ]:
out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_path = out_dir / f'litdata_streaming_{stamp}.json'
payload = {
    'benchmark': 'litdata_streaming',
    'timestamp_utc': stamp,
    's3_bucket': S3_BUCKET,
    's3_prefix': S3_PREFIX,
    'split': SPLIT,
    'batch_size': BATCH_SIZE,
    'warmup_batches': WARMUP_BATCHES,
    'measure_batches': MEASURE_BATCHES,
    'result': result,
}
out_path.write_text(json.dumps(payload, indent=2))
print('Wrote:', out_path)
